# Procura informada — Grupo 10

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.model import Problem
from src.planning import Planner
from src.informed import search
from src.local_search import hill_climbing
from src.csp import solve_csp
from time import perf_counter
problem = Problem(ROOT / 'data_tiny')
t0 = perf_counter()
planner = Planner(problem)
preparation_seconds = perf_counter() - t0
print(f'Preparação comum: {preparation_seconds:.4f} s')
print('Lotes por onda:', [len(w) for w in planner.waves])
print('Valores por domínio:', [len(o) for o in planner.options])

## Formulação e limites da pesquisa

Usamos o custo total em **quilómetros**. Cada linha de `order_items` é indivisível, mas linhas da mesma encomenda podem viajar separadamente. Não inventamos exigências de refrigeração, turnos ou limites de cais que não constam dos dados. A matriz fornecida determina distância e tempo; não usamos serviços externos.

Para tornar a comparação executável, restringimos o espaço de pesquisa a **cinco ondas**: recolhas de 30/10, entregas de 30/10, recolhas de 31/10, entregas de 31/10 recolhidas na véspera e restantes entregas de 31/10. Cada veículo executa as ondas por esta ordem, com relógio próprio; não precisa de esperar que o outro termine a onda. Itens do mesmo produtor/dia são agrupados; entregas são agrupadas por destino/dia e por recolha na véspera ou no próprio dia. A janela do lote é a interseção das janelas dos seus itens. Um lote não é dividido entre veículos. Se a interseção for vazia, o programa informa que esta formulação exige divisão do lote.

Uma decisão escolhe a distribuição dos lotes da onda pelos dois veículos e a ordem das visitas. Cada veículo faz no máximo uma viagem por onda, partindo e regressando ao armazém. Nas recolhas descarrega tudo ao regressar; nas entregas carrega antes de partir. Um veículo pode ficar parado numa onda. Estas escolhas são **hipóteses de implementação**, não exigências adicionais do enunciado.

O estado contém a onda seguinte, as duas rotas parciais, o instante de disponibilidade de cada veículo e os instantes de chegada e descarga de cada item no armazém. A disponibilidade para carregamento começa no fim da descarga. A permanência máxima de 24 h conta desde a chegada ao armazém até ao início do carregamento. Cada visita a produtor/cliente demora 10 min e cada carga/descarga demora 15 min. As janelas referem-se ao início do serviço. Esperas são permitidas; no carregamento esperamos pela partida útil para a primeira entrega.

Os sucessores enumeram todas as permutações e todos os cortes em duas sequências, rejeitando excessos de capacidade e violações temporais. O objetivo exige todos os itens entregues, regresso ao armazém e validação integral por `Problem.validate(require_all=True)`, incluindo inventário do armazém e precedências. A capacidade do armazém é verificada na solução completa, não usada como poda intermédia.

**Limite:** as conclusões de otimalidade de A* referem-se apenas a este espaço de ondas, lotes e horários determinísticos. Não se demonstra o ótimo do problema geral; falhar aqui também não prova que o problema geral é inviável. `data_small` permanece na EDA; esta comparação usa os dois veículos de `data_tiny`, como solicitado.

## Procura informada: Greedy e A*

`g(n)` é a distância das ondas já decididas. Para cada onda futura calculamos dois limites inferiores:

- **h1:** maior distância de ida e volta ao armazém para um dos seus lotes, limitada superiormente pelo mínimo geométrico da onda. Somamos estes valores nas ondas restantes.
- **h2:** custo mínimo entre todas as distribuições e ordens de uma onda, respeitando capacidade, mas ignorando horários, stock e ligação temporal às outras ondas. Somamos estes mínimos nas ondas restantes.

A limitação em h1 evita depender de desigualdades triangulares exatas numa matriz arredondada. Por construção, `0 ≤ h1 ≤ h2 ≤ custo restante` no espaço definido. Ambas são consistentes: a onda escolhida custa pelo menos o respetivo limite. h2 é mais informativa, mas exige enumerar as opções na preparação comum. Essa preparação é medida separadamente e usada em todas as execuções.

**Greedy Best-First** usa apenas `f=h`; **A*** usa `f=g+h`. A fila é um heap. Nos empates preferimos maior profundidade e, depois, a ordem de geração. Como as heurísticas dependem da onda e não da distribuição já escolhida, Greedy pode produzir exatamente a mesma solução com h1 e h2; esta é uma limitação destas heurísticas para Greedy. A* usa também o custo acumulado.

Contamos como expandido um nó não terminal retirado da fila cujos sucessores são examinados. Nós gerados incluem a raiz e os sucessores viáveis inseridos na fila. Não há fusão de estados nem conjunto fechado: pesquisamos uma árvore finita de profundidade cinco. O limite de 20 000 expansões é indicado como interrupção, nunca como solução ou prova de inviabilidade.

In [ ]:
informed_results = [search(planner, algorithm, heuristic)
                    for algorithm in ('greedy', 'astar') for heuristic in ('h1', 'h2')]
for result in informed_results:
    assert result.routes is not None, (result.algorithm, result.status)
    check = problem.validate(result.routes)
    assert check.feasible, check.errors
rows = [dict(metodo=r.algorithm, km=r.cost_km, nos_expandidos=r.expanded,
             nos_gerados=r.generated, segundos=r.seconds, estado=r.status)
        for r in informed_results]
display(pd.DataFrame(rows).round(4))

In [ ]:
for result in informed_results:
    print(result.algorithm)
    display(problem.validate(result.routes).timeline)